# EKF con azúcar total · dos ajustes de biomasa del notebook 25

Reproducción cronológica de LAB016–024 con el EKF de seis estados del notebook 06. El modelo contiene biomasa viable, YAN, azúcar total, etanol, CO₂ disuelto y el acumulador de flujo de la ventana. Se cargan los dos vectores químicos completos del notebook 25 ejecutado. Se aplican la misma solubilidad variable, los mismos coeficientes de gas, los mismos pulsos y las mismas covarianzas de partida. La covarianza de proceso usa las intensidades del 06 y su mapeo depende del estado y de los rendimientos de cada ajuste.

La temperatura es la mediana del último minuto completado. El filtro recibe el flujo medio de cada ventana cerrada de 30 min con al menos 30 lecturas y un hueco máximo de dos minutos, incluidos los extremos. Se corrige cuando esa ventana y la inmediatamente anterior son válidas y alcanzan 0,05 g/(L·h). Se usan lecturas observadas. LAB022–024 empiezan a 0,25 h, por lo que la primera ventana queda incompleta. Las muestras de biomasa posteriores a la inicial se reservan para la evaluación.

R se fija en (0,04 g/(L·h))² como en el notebook 06. Los valores de σX pertenecen a la calibración química del 25.

In [ ]:
from pathlib import Path
from dataclasses import replace
import ast
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd

ROOT=next(path for path in [Path.cwd(),*Path.cwd().parents] if (path/'src/wine_fermentation').is_dir())
sys.path.insert(0,str(ROOT/'src'))
sys.dont_write_bytecode=True
from wine_fermentation import model as kin
from wine_fermentation.comparison import chemical_observations
from wine_fermentation.models.historical.calibration import load

diagnostics=ROOT/'notebooks/diagnostics'
fit_source=diagnostics/'25_zenteno_azucar_total_temperatura_causal.source.ipynb'
fit_path=fit_source.with_name(fit_source.name.replace('.source.','.executed.'))
fit_nb=nbformat.read(fit_path,4)
inputs_nb=nbformat.read(fit_source,4)
assert [c.source for c in fit_nb.cells]==[c.source for c in inputs_nb.cells]
tag='CHEMICAL_PARAMETERS_JSON '
[payload]=[line[len(tag):] for cell in fit_nb.cells for item in cell.get('outputs',[])
           if item.output_type=='stream' for line in item.text.splitlines() if line.startswith(tag)]
fit_parameters=json.loads(payload)
parameters={f'σX = {weight}': fit_parameters[f'Refit σX = {weight}'] for weight in ('0.50','0.25')}
gas_assignment=next(line for cell in inputs_nb.cells for line in cell.source.splitlines()
                    if line.startswith('KLA,GAIN='))
KLA,GAIN=ast.literal_eval(gas_assignment.split('=',1)[1])
assert all(parameters['σX = 0.50'][name]==parameters['σX = 0.25'][name]
           for name in ('Kie0','Kn0','Kd0','m0','Yxs'))
display(pd.DataFrame(parameters))
print('kLa y ganancia',KLA,GAIN)
print('SHA-256 de parámetros y salidas del 25',hashlib.sha256(fit_path.read_bytes()).hexdigest())

def definitions(notebook,names):
    result={}
    for cell in notebook.cells:
        if cell.cell_type=='code':
            for node in ast.parse(cell.source).body:
                if isinstance(node,ast.FunctionDef) and node.name in names:
                    result[node.name]=ast.get_source_segment(cell.source,node)
    assert result.keys()==set(names)
    return result

thermal_profiles={}
for function in definitions(inputs_nb,['register_temperature','causal_temperature']).values():
    exec(compile(function,str(fit_source),'exec'))


## Entradas comunes y ventanas de CO₂

Se reproducen los iniciales, pulsos y horizontes del 25. En todos los lotes, la selección de ventanas se calcula una sola vez a partir del registro de flujo. Las dos estimaciones reciben esos mismos valores en las mismas horas.

In [ ]:
LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018','LAB022','LAB023','LAB024']
historical=LABS[:6]
gas_calibration=LABS[:3]
v=load(ROOT)
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',
                usecols=['lab','time_h','record_kind','remaining_volume_L'])
batches={}
volumes={}

def volume_function(event_times,event_volumes,initial):
    def volume(times):
        return np.r_[initial,event_volumes][np.searchsorted(event_times,np.asarray(times),side='right')]
    return volume

for lab in historical:
    batch=v.batches[lab]
    events=raw.loc[raw.lab.eq(lab)&raw.record_kind.eq('withdrawal_after')].sort_values('time_h')
    volume=volume_function(events.time_h.to_numpy(),events.remaining_volume_L.to_numpy(),2.)
    volumes[lab]=volume
    if lab in gas_calibration:
        batch=replace(batch,pulses={**batch.pulses,'N':tuple((t,d*2/float(volume(t))) for t,d in batch.pulses['N'])})
    if lab=='LAB017':
        schedule=batch.pulses['N']
        batch=replace(batch,pulses={**batch.pulses,'N':((min(t for t,_ in schedule),sum(d for _,d in schedule)),)})
    if lab=='LAB021':
        batch=replace(batch,initials={**batch.initials,'X':.27})
    batches[lab]=batch
    register_temperature(lab,v.raw_temperature[lab])

# Reuse the recent batch preparation, which defines the same sample support and initial values.
[recent_inputs]=[cell.source for cell in inputs_nb.cells if 'def read_holdout_backup(' in cell.source]
recent=recent_inputs[:recent_inputs.index('    raw = holdout_raw_flow')]
recent+='    volumes[lab]=volume_function(withdrawals,-.05*np.arange(1,len(withdrawals)+1)+initial_volume,initial_volume)\n'
exec(compile(recent,str(fit_source),'exec'))
batches.update(holdout_batches)
assert set(batches)==set(LABS)

def sensor_windows(raw_flow,volume,start,end):
    raw_flow=raw_flow.dropna(subset=['t_h','flow_sccm']).sort_values('t_h')
    ids=np.ceil(raw_flow.t_h/.5-1e-10).astype(int)
    grouped={key: frame for key,frame in raw_flow.groupby(ids)}
    rows=[]
    for key in range(1,int(np.floor(end/.5+1e-10))+1):
        stop=key*.5
        window=grouped.get(key)
        valid=False
        measured=np.nan
        if window is not None and len(window):
            times=window.t_h.to_numpy()
            valid=(stop-.5>=start and len(times)>=30 and
                   np.diff(np.r_[stop-.5,times,stop]).max()<=2/60+1e-10)
            if valid:
                measured=float(np.mean(window.flow_sccm.to_numpy()*mass_factor/volume(times)))
        rows.append((stop,measured,valid))
    sensor=pd.DataFrame(rows,columns=['t_h','measured','valid']).set_index('t_h')
    usable=sensor.valid&sensor.measured.ge(.05)
    sensor['update']=usable&usable.shift(fill_value=False)
    return sensor

data={}
scoring_samples={}
support_audit=[]
for lab,batch in batches.items():
    start=float(batch.time[0])
    end=float(v.horizons[lab]) if lab in historical else holdout_horizon
    if lab in historical:
        flow=v.raw_co2[lab]
        samples=(chemical_observations(v.chem,lab,'X') if lab in gas_calibration
                 else v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h','X_viable_g_l']))
    else:
        flow=holdout_raw_flow.loc[holdout_raw_flow.lab.eq(lab)].rename(columns={'h':'t_h'})
        samples=holdout_biomass.loc[holdout_biomass.lab.eq(lab)&holdout_biomass['sample'].ne('01')]
        samples=samples.rename(columns={'hours':'t_h','X_g_L':'X_viable_g_l'})
    samples=samples.loc[(samples.t_h>start)&(samples.t_h<=end),['t_h','X_viable_g_l']]
    scoring_samples[lab]=samples.sort_values('t_h').reset_index(drop=True)
    sensor=sensor_windows(flow,volumes[lab],start,end)
    assert sensor['update'].any() and len(samples)>0
    initial=batch.initials
    x0=np.array([initial['X'],initial['N'],initial['G']+initial['F'],initial['E'],0.,0.])
    P0=np.diag([max(.03,.2*x0[0])**2,.015**2,25.,4.,0.,0.])
    pulses={round(t,10): dose for t,dose in batch.pulses['N'] if start<t<=end}
    grid=np.unique(np.round(np.r_[np.arange(start,end,1/12),list(pulses),sensor.index,end],10))
    grid=grid[grid>=start-1e-10]
    assert np.isclose(grid[0],start)
    assert thermal_profiles[lab][0][0]<=start
    data[lab]=dict(grid=grid,initial=x0,P0=P0,pulses=pulses,sensor=sensor,batch=batch)
    support_audit.append(dict(LAB=lab,inicio=start,fin=end,n_biomasa=len(samples),
                              ventanas_válidas=int(sensor.valid.sum()),correcciones=int(sensor['update'].sum())))
display(pd.DataFrame(support_audit).set_index('LAB'))


## Reproducción del EKF

Se reutilizan las tasas, transferencia de gas, diferencias finitas y ruido de proceso del EKF-S del notebook 06. RK4 integra hasta el siguiente minuto, pulso o cierre de ventana. La temperatura de cada intervalo es la disponible al comienzo de ese intervalo. La actualización usa la forma de Joseph y reinicia el acumulador en todos los cierres. Para evaluar una muestra entre pasos se propaga desde el último estado disponible hasta su hora.

In [ ]:
from types import FunctionType

ekf_path=ROOT/'notebooks/EKF/06_zenteno_single_sugar_variable_solubility.source.ipynb'
ekf_source=nbformat.read(ekf_path,4)
[s_cell]=[cell.source for cell in ekf_source.cells if "PS={'mu0'" in cell.source]
s_functions=definitions(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(s_cell)]),
                        ['rates_ekf','jacobian','noise_matrix'])
gas_functions=definitions(ekf_source,['co2_saturation','gas_exchange'])
SCALE_EKF=np.array([.3,.25,160.,15.,.2,1.])
R_CO2=.04**2
H=np.array([0.,0.,0.,0.,0.,2.])
IDENTITY=np.eye(6)

def thermal_steps(batch,start,end):
    available=thermal_profiles[batch.batch][0]
    left=np.searchsorted(available,start+1e-10,side='right')
    right=np.searchsorted(available,end-1e-10,side='left')
    edges=np.r_[start,available[left:right],end]
    return tuple(zip(edges[:-1],np.diff(edges),causal_temperature(batch,edges[:-1]+1e-10)))

def make_model(p):
    namespace=dict(np=np,KLA=KLA,GAIN=GAIN,PCO2_BAR=1.,PS=p,theta=p,YXS=p['Yxs'],SCALE_EKF=SCALE_EKF)
    for function in [*gas_functions.values(),*s_functions.values()]:
        exec(compile(function,str(ekf_path),'exec'),namespace)
    rates=namespace['rates_ekf']
    def transition(z,start,end,steps,S0):
        w=np.atleast_2d(z)*SCALE_EKF
        for time,dt,temp in steps:
            # Split long gaps too, even when the held temperature stays unchanged.
            count=max(1,int(np.ceil(dt*60-1e-7)))
            h=dt/count
            for j in range(count):
                t=time+j*h
                a=rates(w,t,temp,S0)
                b=rates(w+h*a/2,t+h/2,temp,S0)
                c=rates(w+h*b/2,t+h/2,temp,S0)
                d=rates(w+h*c,t+h,temp,S0)
                w=np.maximum(w+h*(a+2*b+2*c+d)/6,0.)
        return w/SCALE_EKF
    namespace['transition']=transition
    return transition,namespace['jacobian'],namespace['noise_matrix']

def replay(inputs,model):
    advance,jacobian,noise=model
    grid=inputs['grid']
    batch=inputs['batch']
    sensor=inputs['sensor']
    windows={t: (row.measured,row.valid,row['update']) for t,row in sensor.iterrows()}
    y0=inputs['initial']
    S0=y0[2]
    z=y0/SCALE_EKF
    free=z.copy()
    P=inputs['P0']/np.outer(SCALE_EKF,SCALE_EKF)
    states=[]
    readings=[]
    for i,time in enumerate(grid):
        if i:
            start=grid[i-1]
            steps=thermal_steps(batch,start,time)
            process=noise(z*SCALE_EKF,steps[0][2],S0)*(time-start)
            z,J=jacobian(z,start,time,steps,S0)
            P=J@P@J.T+process
            P=(P+P.T)/2
            free=advance(free,start,time,steps,S0)[0]
        if time in inputs['pulses']:
            dose=inputs['pulses'][time]/SCALE_EKF[1]
            z[1]+=dose
            free[1]+=dose
        if time in windows:
            measured,valid,update=windows[time]
            predicted=float(H@z)
            nis=np.nan
            clipped=0
            if update:
                variance=float(H@P@H+R_CO2)
                assert variance>0
                gain=P@H/variance
                innovation=measured-predicted
                updated=z+gain*innovation
                clipped=int(np.count_nonzero(updated[:5]<0))
                z=np.maximum(updated,0.)
                B=IDENTITY-np.outer(gain,H)
                P=B@P@B.T+np.outer(gain,gain)*R_CO2
                P=(P+P.T)/2
                nis=innovation**2/variance
            readings.append((time,measured,valid,update,predicted,float(H@free),nis,clipped))
            z[5]=free[5]=0.
            P[5,:]=P[:,5]=0.
        assert np.isfinite(z).all() and np.linalg.eigvalsh(P).min()>=-1e-7
        states.append(np.stack([z,free]))
    readings=pd.DataFrame(readings,columns=['t_h','measured','valid','update','forecast','free','NIS','recortes'])
    np.testing.assert_array_equal(readings['update'],sensor['update'])
    return np.stack(states),readings

models={label:make_model(vector) for label,vector in parameters.items()}
results={}
for label,model in models.items():
    results[label]={}
    for lab,inputs in data.items():
        results[label][lab]=replay(inputs,model)
        print(label,lab,'completado',flush=True)


## Biomasa en los mismos nueve holdouts

El RMSE y el sesgo se calculan con muestras posteriores a la inicial. La columna «Libre» usa la misma dinámica y temperatura con la corrección desactivada. El exceso medio promedia la parte positiva del error en todas las muestras. Los promedios dan el mismo peso a cada lote.

In [ ]:
labels=list(parameters)
styles={labels[0]:dict(color='#D55E00'),labels[1]:dict(color='#0072B2')}

def states_at_samples(inputs,model,states,times):
    grid=inputs['grid']
    advance=model[0]
    out=[]
    for time in times:
        index=np.searchsorted(grid,time+1e-10,side='right')-1
        assert index>=0
        last=grid[index]
        points=states[index]
        if time>last+1e-10:
            points=advance(points,last,time,thermal_steps(inputs['batch'],last,time),inputs['initial'][2])
        out.append(points[:,0]*SCALE_EKF[0])
    return np.array(out)

rows=[]
for lab in LABS:
    samples=scoring_samples[lab]
    y=samples.X_viable_g_l.to_numpy()
    for label in labels:
        predictions=states_at_samples(data[lab],models[label],results[label][lab][0],samples.t_h)
        for index,mode in enumerate(('EKF','Libre')):
            error=predictions[:,index]-y
            rows.append(dict(LAB=lab,ajuste=label,modo=mode,n=len(y),
                             RMSE=np.sqrt(np.mean(error**2)),MAE=np.mean(abs(error)),
                             sesgo=np.mean(error),exceso=np.maximum(error,0.).mean()))
metrics=pd.DataFrame(rows)
wide=metrics.pivot(index='LAB',columns=['modo','ajuste'],values='RMSE')
display(wide.reindex(LABS).round(4))
averages=metrics.groupby(['modo','ajuste'])[['RMSE','MAE','sesgo','exceso']].mean()
display(averages.round(4))
print('BIOMASS_BY_LAB',metrics.to_json(orient='records'))
print('BIOMASS_AVERAGES',averages.reset_index().to_json(orient='records'))

def biomass_panels(mode_index,title):
    fig,axes=plt.subplots(3,3,figsize=(15,11),layout='constrained')
    for ax,lab in zip(axes.flat,LABS):
        inputs=data[lab]
        grid=inputs['grid']
        samples=scoring_samples[lab]
        for label in labels:
            scores=metrics.loc[metrics.LAB.eq(lab)&metrics.ajuste.eq(label)&metrics.modo.eq(('EKF','Libre')[mode_index])]
            score=scores.RMSE.iloc[0]
            x=results[label][lab][0][:,mode_index,0]*SCALE_EKF[0]
            ax.plot(grid,x,label=f'{label} · RMSE {score:.3f}',**styles[label])
        ax.scatter(samples.t_h,samples.X_viable_g_l,color='black',s=22,zorder=4,label='Muestras')
        ax.scatter(grid[0],inputs['initial'][0],facecolors='none',edgecolors='black',s=28,label='Inicial')
        for pulse in inputs['pulses']:
            ax.axvline(pulse,c='#888888',ls=':',lw=.8)
        ax.set(title=lab,xlabel='Tiempo [h]',ylabel='Biomasa [g/L]',ylim=(0,None))
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.show()

biomass_panels(0,'Biomasa estimada con el EKF')
biomass_panels(1,'Biomasa en lazo abierto')


## Pronóstico de CO₂ y correcciones

El RMSE usa el pronóstico previo a la actualización en las ventanas asimiladas. El NIS es el cuadrado de la innovación dividido por su varianza predicha. Los recortes cuentan componentes de estado negativos tras la corrección.

In [ ]:
gas_metrics=[]
fig,axes=plt.subplots(3,3,figsize=(15,11),layout='constrained')
for ax,lab in zip(axes.flat,LABS):
    sensor=data[lab]['sensor']
    ax.plot(sensor.index,sensor.measured,c='black',lw=.8,label='Medido')
    for label in labels:
        readings=results[label][lab][1]
        active=readings.loc[readings['update']]
        error=active.forecast-active.measured
        gas_metrics.append(dict(LAB=lab,ajuste=label,n=len(active),RMSE=np.sqrt(np.mean(error**2)),
                                sesgo=error.mean(),NIS_mediano=active.NIS.median(),
                                recortes=int(active.recortes.sum())))
        ax.plot(readings.t_h,readings.forecast,label=label,**styles[label])
    ax.set(title=lab,xlabel='Tiempo [h]',ylabel='Flujo de CO₂ [g/(L·h)]',ylim=(0,None))
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
fig.suptitle('Flujo de CO₂ pronosticado antes de corregir')
plt.show()
gas_metrics=pd.DataFrame(gas_metrics)
assert gas_metrics.groupby('LAB').n.nunique().eq(1).all()
display(gas_metrics.round(4))
print('CO2_AVERAGES',gas_metrics.groupby('ajuste')[['RMSE','sesgo','recortes']].mean().reset_index().to_json(orient='records'))
